# Premier League Prediction V2

## 1. Imports and Global Configuration

In [15]:
import re
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score, log_loss
from sklearn.model_selection import TimeSeriesSplit
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

DATA_ROOT = Path(
    "/mnt/d/coding/year3/term2/data_science/premier_league_prediction/data"
)
USE_ADDITIONAL_DATA = True
ROLLING_WINDOWS = (3, 5, 10)
ROLLING_WINDOW = 5
ROLLING_DECAY = 0.75

COUNTRY_CODES = {
    "eng",
    "es",
    "de",
    "fr",
    "it",
    "nl",
    "pt",
    "ru",
    "ua",
    "gr",
    "be",
    "at",
    "hr",
    "rs",
    "tr",
    "ch",
    "dk",
    "no",
    "se",
    "pl",
    "cz",
    "sk",
}

TEAM_ALIASES = {
    "man city": "manchester city",
    "man utd": "manchester united",
    "manchester utd": "manchester united",
    "spurs": "tottenham hotspur",
    "tottenham": "tottenham hotspur",
    "wolves": "wolverhampton wanderers",
    "leicester": "leicester city",
    "norwich": "norwich city",
    "west ham": "west ham united",
    "newcastle": "newcastle united",
    "brighton": "brighton and hove albion",
}

COMP_WEIGHTS = {
    "ucl": {"group": 1.0, "knockout": 2.0, "late": 3.0},
    "uel": {"group": 0.8, "knockout": 1.5, "late": 2.0},
    "fa": {"early": 0.5, "late": 1.5},
    "carabao": {"early": 0.3, "late": 1.0},
}

TEAM_BOOST_OVERRIDES = {
    "liverpool": 0.40,
    "manchester city": 0.00,
}

CONTEXT_COLS = {
    "Possession": ["Possession"],
    "Shot_Creating_Actions": ["Shot_Creating_Actions", "SCA"],
    "Successful_Dribbles": ["Successful_Dribbles", "Dribbles"],
    "Final_Third_Entries": ["Final_Third_Entries"],
    "Final_Third_Entries_Allowed": ["Final_Third_Entries_Allowed"],
    "Aerial_Battles_Won_Pct": ["Aerial_Battles_Won%", "Aerial_Battles_Won_Pct"],
    "Save_Pct": ["Save%", "Save_Pct"],
    "PPDA": ["PPDA"],
    "Allowed_PPDA": ["Allowed_PPDA"],
}

## 2. Utility Helpers (Logging, Column Resolution, Team Normalization)

In [16]:
def log(msg: str) -> None:
    print(f"[INFO] {msg}")


def find_col(df: pd.DataFrame, *candidates: str) -> Optional[str]:
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None


def normalize_team(name) -> str:
    s = re.sub(r"[^a-z0-9\s]", " ", str(name).strip().lower())
    tokens = [t for t in s.split() if t]
    if tokens and tokens[0] in COUNTRY_CODES:
        tokens = tokens[1:]
    if tokens and tokens[-1] in COUNTRY_CODES:
        tokens = tokens[:-1]
    s = " ".join(tokens)
    return TEAM_ALIASES.get(s, s)


def col_or_nan(df: pd.DataFrame, col: str) -> pd.Series:
    if col in df.columns:
        return df[col]
    return pd.Series(np.nan, index=df.index)

## 3. Data Loading (League, Positions, Cup/European Fixtures)

In [17]:
def load_premier_league(base: Path) -> pd.DataFrame:
    df = pd.read_csv(base / "premier_league.csv", low_memory=False)
    if USE_ADDITIONAL_DATA:
        extra = base / "additional_data" / "premier_league.csv"
        extra_df = pd.read_csv(extra, low_memory=False)
        common = list(set(df.columns) & set(extra_df.columns))
        df = (
            pd.concat([df[common], extra_df[common]], ignore_index=True)
            .drop_duplicates()
            .copy()
        )
        log("Loaded additional_data/premier_league.csv")
    log(f"Raw rows loaded: {len(df):,}")
    return df


def parse_home_away(df: pd.DataFrame) -> pd.Series:
    for col in ("h_a", "side"):
        if col in df.columns:
            return df[col].astype(str).str.lower().str.strip()
    if "Venue" in df.columns:
        return (
            df["Venue"]
            .astype(str)
            .str.lower()
            .str.strip()
            .map({"home": "h", "away": "a"})
        )
    return pd.Series(["a"] * len(df), index=df.index)


def load_position_map(path: Path) -> dict:
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    team_col = find_col(df, "team", "teamid", "club")
    pos_col = find_col(df, "position", "pos", "rank")
    if not team_col or not pos_col:
        raise ValueError(f"{path} must have Team and Position columns")
    df["team_norm"] = df[team_col].map(normalize_team)
    df["pos"] = pd.to_numeric(df[pos_col], errors="coerce")
    df = df.dropna(subset=["team_norm", "pos"])
    return dict(zip(df["team_norm"], df["pos"]))


def _infer_stage(comp: str, stage_text: str, dt: pd.Timestamp) -> str:
    s = str(stage_text).lower()
    if any(
        k in s for k in ["semi", "final", "quarter", "qf", "sf", "r16", "round of 16"]
    ):
        return (
            "late"
            if any(k in s for k in ["semi", "final", "quarter", "qf", "sf"])
            else "knockout"
        )
    if "group" in s:
        return "group"
    month = dt.month if pd.notna(dt) else 1
    if comp in ("ucl", "uel"):
        return (
            "group"
            if month in [9, 10, 11, 12]
            else ("late" if month == 8 else "knockout")
        )
    if comp in ("fa", "carabao"):
        return "late" if month >= 3 else "early"
    return "early"


def load_competition_matches(path: Path, comp: str) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=str)
    df.columns = df.columns.str.strip()
    date_col = find_col(df, "date")
    home_col = find_col(df, "home team", "home_team", "home")
    away_col = find_col(df, "away team", "away_team", "away")
    stage_col = find_col(df, "stage", "round")
    if not all([date_col, home_col, away_col]):
        raise ValueError(f"{path.name} is missing required columns")
    date_text = df[date_col].astype(str).str.strip()
    date_text = date_text.str.replace(
        r"([A-Za-z]+\s+\d{1,2})-\d{1,2}(\s+\d{4})", r"\1\2", regex=True
    )
    df["_date"] = pd.to_datetime(date_text, errors="coerce")
    df["_stage"] = df[stage_col] if stage_col else ""
    rows = []
    for _, row in df.iterrows():
        if pd.isna(row["_date"]):
            continue
        stage = _infer_stage(comp, row["_stage"], row["_date"])
        weight = COMP_WEIGHTS.get(comp, {}).get(stage, 0.0)
        for col in (home_col, away_col):
            rows.append(
                {
                    "team": normalize_team(row[col]),
                    "date": row["_date"],
                    "weight": weight,
                }
            )
    return pd.DataFrame(rows, columns=["team", "date", "weight"])


def _load_competition_events(data_root: Path) -> pd.DataFrame:
    if not USE_ADDITIONAL_DATA:
        return pd.DataFrame(columns=["team", "date", "weight"])
    additional = data_root / "additional_data"
    return pd.concat(
        [
            load_competition_matches(additional / "champion_league.csv", "ucl"),
            load_competition_matches(additional / "europa_league.csv", "uel"),
            load_competition_matches(additional / "fa_cup.csv", "fa"),
            load_competition_matches(additional / "carabao.csv", "carabao"),
        ],
        ignore_index=True,
    )

## 4. Base Normalization and Trainable Row Filtering

In [18]:
def _normalize_base_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["Date"] = pd.to_datetime(out["Date"], errors="coerce")
    out["Result"] = out["Result"].astype(str).str.lower().str.strip()
    out["Team"] = out["Team"].map(normalize_team)
    out["Opponent"] = out["Opponent"].map(normalize_team)
    out["xG"] = pd.to_numeric(out["xG"], errors="coerce")
    out["xGA"] = pd.to_numeric(out["xGA"], errors="coerce")
    out["home_advantage"] = (
        parse_home_away(out).replace({"home": "h", "away": "a"}).fillna("a")
    )
    return out


def _add_rolling_helper_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["_scored"] = pd.to_numeric(col_or_nan(out, "Scored"), errors="coerce")
    out["_conceded"] = pd.to_numeric(col_or_nan(out, "Conceded"), errors="coerce")
    out["_win_val"] = out["Result"].map({"w": 1.0, "d": 0.5, "l": 0.0})
    out["_ppda"] = pd.to_numeric(
        out.get("PPDA", pd.Series(dtype=float)), errors="coerce"
    )
    if "xpts" in out.columns:
        out["_xpts"] = pd.to_numeric(out["xpts"], errors="coerce")
    return out


def _filter_trainable_rows(df: pd.DataFrame) -> pd.DataFrame:
    filtered = df[
        df["Date"].notna()
        & df["Result"].isin(["w", "d", "l"])
        & df["Team"].notna()
        & df["Opponent"].notna()
    ].copy()
    log(f"Rows after filtering: {len(filtered):,}")
    return filtered


def _drop_rolling_helper_columns(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop(
        columns=["_scored", "_conceded", "_win_val", "_ppda", "_xpts"], errors="ignore"
    ).copy()

## 5. Chronological Train/Validation/Test Split

In [19]:
def time_based_split(df, val_frac=0.20, test_frac=0.20, random_state=42):
    if val_frac <= 0 or test_frac <= 0 or val_frac + test_frac >= 1.0:
        raise ValueError("Invalid split fractions")
    ordered = df.copy()
    ordered["Date"] = pd.to_datetime(ordered["Date"], errors="coerce")
    ordered = (
        ordered[ordered["Date"].notna()].sort_values("Date").reset_index(drop=True)
    )
    if ordered.empty:
        raise ValueError("No dated rows available")

    unique_dates = pd.Index(ordered["Date"].unique()).sort_values()
    n = len(unique_dates)
    n_test = max(1, int(round(n * test_frac)))
    n_val = max(1, int(round(n * val_frac)))
    n_train = n - n_val - n_test
    if n_train < 1:
        raise ValueError("Split fractions leave no training dates")

    train_dates = set(unique_dates[:n_train])
    val_dates = set(unique_dates[n_train : n_train + n_val])
    test_dates = set(unique_dates[n_train + n_val :])

    train_df = (
        ordered[ordered["Date"].isin(train_dates)]
        .copy()
        .sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
    )
    val_df = (
        ordered[ordered["Date"].isin(val_dates)]
        .copy()
        .sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
    )
    test_df = (
        ordered[ordered["Date"].isin(test_dates)]
        .copy()
        .sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
    )

    assert (
        pd.to_datetime(train_df["Date"]).max() < pd.to_datetime(val_df["Date"]).min()
    ), "Train/val date overlap"
    assert (
        pd.to_datetime(val_df["Date"]).max() < pd.to_datetime(test_df["Date"]).min()
    ), "Val/test date overlap"
    log(f"Split: train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")
    return train_df, val_df, test_df

## 6. Rolling and Weighted Lagged Feature Builders

In [20]:
def _lagged_rolling_mean(series: pd.Series, window: int) -> pd.Series:
    return series.shift(1).rolling(window, min_periods=1).mean()


def _lagged_weighted_mean(
    series: pd.Series, window: int, decay: float = ROLLING_DECAY
) -> pd.Series:
    shifted = pd.to_numeric(series, errors="coerce").shift(1)

    def _weighted_mean(values):
        values = np.asarray(values, dtype=float)
        valid = np.isfinite(values)
        if not valid.any():
            return float(np.nan)
        values = values[valid]
        weights = np.array(
            [decay ** (len(values) - i - 1) for i in range(len(values))], dtype=float
        )
        weights /= weights.sum()
        return float(np.dot(values, weights))

    return shifted.rolling(window, min_periods=1).apply(_weighted_mean, raw=True)


def build_rolling_features(
    df: pd.DataFrame, windows: tuple = ROLLING_WINDOWS
) -> pd.DataFrame:
    df = df.sort_values("Date").copy()
    grp = df.groupby("Team", group_keys=False)
    metrics = {
        "xG": "xG",
        "xGA": "xGA",
        "scored": "_scored",
        "conceded": "_conceded",
        "win_rate": "_win_val",
        "ppda": "_ppda",
    }
    if "_xpts" in df.columns:
        metrics["xpts"] = "_xpts"

    for window in windows:
        for label, column in metrics.items():
            df[f"rolling_{label}_{window}"] = grp[column].transform(
                lambda s, w=window: _lagged_rolling_mean(s, w)
            )
            df[f"weighted_{label}_{window}"] = grp[column].transform(
                lambda s, w=window: _lagged_weighted_mean(s, w)
            )
        df[f"rolling_goal_diff_{window}"] = (
            df[f"rolling_scored_{window}"] - df[f"rolling_conceded_{window}"]
        )
        df[f"weighted_goal_diff_{window}"] = (
            df[f"weighted_scored_{window}"] - df[f"weighted_conceded_{window}"]
        )

    log(f"Rolling features built (windows={windows})")
    return df

## 7. Leakage-Safe Feature Attachments (Fatigue, Opponent Rolling, H2H, Referee, Context, Motivation)

In [21]:
def attach_opponent_rolling_features(
    df: pd.DataFrame, source_df: pd.DataFrame
) -> pd.DataFrame:
    out = df.sort_values(["Date", "Opponent"]).copy()
    rolling_cols = [
        c
        for c in source_df.columns
        if c.startswith("rolling_") or c.startswith("weighted_")
    ]
    if not rolling_cols:
        return out

    source = (
        source_df[["Team", "Date"] + rolling_cols]
        .copy()
        .rename(columns={"Team": "Opponent", **{c: f"opp_{c}" for c in rolling_cols}})
        .sort_values(["Opponent", "Date"])
    )

    merged_parts = []
    for opponent, left_team in out.groupby("Opponent", sort=False):
        right_team = source[source["Opponent"].eq(opponent)]
        if right_team.empty:
            merged_parts.append(left_team)
            continue
        left_team = left_team.sort_values("Date").copy()
        left_team["_orig_idx"] = left_team.index
        merged = pd.merge_asof(
            left_team,
            right_team.drop(columns=["Opponent"]),
            left_on="Date",
            right_on="Date",
            direction="backward",
            allow_exact_matches=False,
        )
        merged_parts.append(merged)

    if merged_parts:
        out = (
            pd.concat(merged_parts)
            .sort_values("_orig_idx")
            .drop(columns=["_orig_idx"], errors="ignore")
        )
    log("Opponent rolling features attached")
    return out


def attach_fatigue(df: pd.DataFrame, comp_df: pd.DataFrame) -> pd.DataFrame:
    comp_df = comp_df.dropna(subset=["team", "date"]).sort_values("date").copy()
    comp_df["date"] = pd.to_datetime(comp_df["date"], errors="coerce").astype(
        "datetime64[ns]"
    )
    comp_df = comp_df.dropna(subset=["date"])
    comp_df["weight"] = pd.to_numeric(comp_df["weight"], errors="coerce").fillna(0.0)
    comp_df["cum_weight"] = comp_df.groupby("team")["weight"].transform(
        lambda s: s.cumsum().shift(1).fillna(0.0)
    )

    left = df.sort_values("Date").reset_index().rename(columns={"index": "_orig_idx"})
    left["Team"] = left["Team"].astype(str)
    left["Date"] = pd.to_datetime(left["Date"], errors="coerce").astype(
        "datetime64[ns]"
    )

    right = comp_df.rename(columns={"team": "Team"}).sort_values("date")
    right["Team"] = right["Team"].astype(str)

    def _window_cum_weight(left_team, right_team, days):
        boundary = left_team[["_orig_idx", "Date"]].copy()
        boundary["Date"] = boundary["Date"] - pd.Timedelta(days=days + 1)
        merged = pd.merge_asof(
            boundary.sort_values("Date"),
            right_team[["date", "cum_weight"]].sort_values("date"),
            left_on="Date",
            right_on="date",
            direction="backward",
            allow_exact_matches=False,
        )
        return merged.set_index("_orig_idx")["cum_weight"].sort_index().fillna(0.0)

    fat_cols = [
        "fatigue_score",
        "fatigue_7d",
        "fatigue_14d",
        "fatigue_30d",
        "fatigue_recent_share",
    ]
    for c in fat_cols:
        left[c] = 0.0

    left_valid = left[left["Date"].notna()].sort_values("Date").copy()
    if not left_valid.empty and not right.empty:
        merged_parts = []
        for team, left_team in left_valid.groupby("Team", sort=False):
            right_team = right[right["Team"].eq(team)]
            if right_team.empty:
                continue
            left_team = left_team.sort_values("Date").copy()
            total = _window_cum_weight(left_team, right_team, days=0)
            fatigue_7d = total - _window_cum_weight(left_team, right_team, days=7)
            fatigue_14d = total - _window_cum_weight(left_team, right_team, days=14)
            fatigue_30d = total - _window_cum_weight(left_team, right_team, days=30)
            tf = pd.DataFrame(
                {
                    "_orig_idx": left_team["_orig_idx"].to_numpy(),
                    "fatigue_7d": fatigue_7d.to_numpy(),
                    "fatigue_14d": fatigue_14d.to_numpy(),
                    "fatigue_30d": fatigue_30d.to_numpy(),
                }
            )
            tf["fatigue_recent_share"] = np.where(
                tf["fatigue_30d"] > 0, tf["fatigue_7d"] / tf["fatigue_30d"], 0.0
            )
            tf["fatigue_score"] = (
                2.0 * tf["fatigue_7d"]
                + 1.1 * (tf["fatigue_14d"] - tf["fatigue_7d"])
                + 0.6 * (tf["fatigue_30d"] - tf["fatigue_14d"])
            )
            merged_parts.append(tf)

        if merged_parts:
            fat_df = pd.concat(merged_parts).set_index("_orig_idx").sort_index()
            left.loc[fat_df.index, fat_cols] = fat_df[fat_cols].to_numpy()

    ordered = left.sort_values("_orig_idx")
    for c in fat_cols:
        df[c] = ordered[c].to_numpy()
    log(f"Fatigue scores attached from {len(comp_df):,} competition rows")
    return df


def attach_position_features(df: pd.DataFrame, pos_map: dict) -> pd.DataFrame:
    df = df.copy()
    df["team_position"] = df["Team"].map(pos_map)
    df["opponent_position"] = df["Opponent"].map(pos_map)
    df["position_gap"] = df["opponent_position"] - df["team_position"]
    return df


def attach_context_features(df: pd.DataFrame) -> tuple:
    added = []
    for canonical, candidates in CONTEXT_COLS.items():
        src = find_col(df, *candidates)
        if src is not None:
            df[canonical] = pd.to_numeric(df[src], errors="coerce")
            added.append(canonical)
    if added:
        log(f"Context features: {added}")
    return df, added


def attach_h2h_features(df: pd.DataFrame, history_df: pd.DataFrame) -> pd.DataFrame:
    H2H_COLS = [
        "h2h_win_rate",
        "h2h_goal_diff",
        "h2h_match_count",
        "h2h_points_per_game",
    ]
    hist = history_df.sort_values("Date").copy()
    hist["_h2h_key"] = hist[["Team", "Opponent"]].apply(
        lambda r: "|".join(sorted([r["Team"], r["Opponent"]])), axis=1
    )
    hist["_h2h_win"] = (hist["Result"] == "w").astype(float)
    hist["_h2h_goal_diff"] = pd.to_numeric(
        col_or_nan(hist, "Scored"), errors="coerce"
    ) - pd.to_numeric(col_or_nan(hist, "Conceded"), errors="coerce")
    hist["_h2h_points"] = hist["Result"].map({"w": 3.0, "d": 1.0, "l": 0.0})
    grp = hist.groupby("_h2h_key", sort=False)
    hist["h2h_win_rate"] = (
        grp["_h2h_win"].transform(lambda s: s.shift(1).expanding().mean()).fillna(0.5)
    )
    hist["h2h_goal_diff"] = (
        grp["_h2h_goal_diff"]
        .transform(lambda s: s.shift(1).expanding().mean())
        .fillna(0.0)
    )
    hist["h2h_match_count"] = grp.cumcount().astype(float)
    hist["h2h_points_per_game"] = (
        grp["_h2h_points"]
        .transform(lambda s: s.shift(1).expanding().mean())
        .fillna(1.0)
    )
    h2h_lookup = (
        hist[["Team", "Opponent", "Date", "_h2h_key"] + H2H_COLS]
        .sort_values("Date")
        .copy()
    )

    out = df.copy()
    out["_h2h_key"] = out[["Team", "Opponent"]].apply(
        lambda r: "|".join(sorted([r["Team"], r["Opponent"]])), axis=1
    )
    for c in H2H_COLS:
        out[c] = np.nan

    merged_parts = []
    for key, left_group in out.groupby("_h2h_key", sort=False):
        right_group = h2h_lookup[h2h_lookup["_h2h_key"].eq(key)][
            ["Date"] + H2H_COLS
        ].sort_values("Date")
        left_group = left_group.sort_values("Date").copy()
        left_group["_orig_idx"] = left_group.index
        if right_group.empty:
            merged_parts.append(left_group)
            continue
        merged = pd.merge_asof(
            left_group,
            right_group,
            on="Date",
            direction="backward",
            allow_exact_matches=False,
            suffixes=("_old", ""),
        )
        for c in H2H_COLS:
            old_col = f"{c}_old"
            if old_col in merged.columns:
                merged.drop(columns=[old_col], inplace=True)
        merged_parts.append(merged)

    if merged_parts:
        out = (
            pd.concat(merged_parts)
            .sort_values("_orig_idx")
            .drop(columns=["_orig_idx", "_h2h_key"], errors="ignore")
        )
    out["h2h_win_rate"] = out["h2h_win_rate"].fillna(0.5)
    out["h2h_goal_diff"] = out["h2h_goal_diff"].fillna(0.0)
    out["h2h_match_count"] = out["h2h_match_count"].fillna(0.0)
    out["h2h_points_per_game"] = out["h2h_points_per_game"].fillna(1.0)
    log("H2H features attached")
    return out


DEFAULT_PPDA = 6.0


def _row_numeric(row: pd.Series, key: str, default: float = np.nan) -> float:
    v = pd.to_numeric(pd.Series([row.get(key, default)]), errors="coerce").iloc[0]
    return default if pd.isna(v) else float(v)


def calculate_recent_form(
    df: pd.DataFrame, team: str, current_date: pd.Timestamp, window: int = 5
) -> dict:
    past = df[(df["Team"].eq(team)) & (df["Date"] < current_date)].sort_values("Date")
    if past.empty:
        return {
            "points_per_game": 0.0,
            "streak": 0,
            "recent_xg": 0.0,
            "recent_goals": 0.0,
        }
    recent = past.tail(window)
    points = recent["Result"].map({"w": 1.0, "d": 0.5, "l": 0.0}).sum()
    ppg = points / len(recent)
    recent_xg = pd.to_numeric(col_or_nan(recent, "xG"), errors="coerce").mean()
    recent_goals = pd.to_numeric(col_or_nan(recent, "Scored"), errors="coerce").mean()
    streak = 0
    for r in recent["Result"].iloc[::-1]:
        if r == "w":
            streak += 1
        elif r == "l":
            streak -= 1
        else:
            break
    return {
        "points_per_game": ppg,
        "streak": streak,
        "recent_xg": recent_xg if pd.notna(recent_xg) else 0.0,
        "recent_goals": recent_goals if pd.notna(recent_goals) else 0.0,
    }


def calculate_opponent_strength(
    df: pd.DataFrame, opponent: str, current_date: pd.Timestamp, window: int = 10
) -> float:
    past = df[(df["Team"].eq(opponent)) & (df["Date"] < current_date)].sort_values(
        "Date"
    )
    if past.empty:
        return 0.5
    opp = past.tail(window).copy()
    opp["_win_val"] = opp["Result"].map({"w": 1.0, "d": 0.5, "l": 0.0})
    return float(opp["_win_val"].mean())


def calculate_fixture_importance(team_pos: float, opponent_pos: float) -> float:
    if pd.isna(team_pos) or pd.isna(opponent_pos):
        return 0.5
    importance = 0.0
    if team_pos <= 2:
        importance += 0.7
    if team_pos >= 16:
        importance += 0.8
    if 3 <= team_pos <= 7:
        importance += 0.5
    if opponent_pos <= 6:
        importance += 0.4
    if abs(team_pos - opponent_pos) <= 2:
        importance += 0.3
    return min(1.0, importance)


def _match_context_features_for_row(
    row, rolling_window, recent_form, opponent_strength
):
    wp = _row_numeric(row, f"weighted_win_rate_{rolling_window}")
    rolling_points = (
        (wp * 3.0)
        if pd.notna(wp)
        else (
            (_row_numeric(row, f"rolling_win_rate_{rolling_window}") * 3.0)
            if pd.notna(_row_numeric(row, f"rolling_win_rate_{rolling_window}"))
            else recent_form["points_per_game"] * 3.0
        )
    )
    xg_edge = _row_numeric(row, f"weighted_xG_{rolling_window}") - _row_numeric(
        row, f"opp_weighted_xG_{rolling_window}"
    )
    goal_diff_edge = _row_numeric(
        row, f"weighted_goal_diff_{rolling_window}"
    ) - _row_numeric(row, f"opp_weighted_goal_diff_{rolling_window}")
    win_rate_edge = _row_numeric(
        row, f"weighted_win_rate_{rolling_window}"
    ) - _row_numeric(row, f"opp_weighted_win_rate_{rolling_window}")
    return {
        "form_vs_strength": recent_form["points_per_game"] - opponent_strength,
        "momentum": rolling_points + recent_form["streak"],
        "xg_edge": 0.0 if pd.isna(xg_edge) else float(xg_edge),
        "goal_diff_edge": 0.0 if pd.isna(goal_diff_edge) else float(goal_diff_edge),
        "win_rate_edge": 0.0 if pd.isna(win_rate_edge) else float(win_rate_edge),
    }


def _referee_pressure_impact_for_row(row):
    is_home = str(row.get("home_advantage", "a")).lower() == "h"
    foul_col = "Home_Fouls" if is_home else "Away_Fouls"
    team_fouls = _row_numeric(row, foul_col, default=0.0)
    ppda_value = _row_numeric(row, "PPDA", default=DEFAULT_PPDA)
    referee_bias = _row_numeric(row, "Referee_Bias_Score", default=0.0)
    aggression = team_fouls + max(0.0, 8.0 - ppda_value)
    return referee_bias * (1.0 + aggression / 10.0)


def attach_referee_features(df: pd.DataFrame, history_df: pd.DataFrame) -> pd.DataFrame:
    def _compute_ref_signal(frame: pd.DataFrame) -> pd.DataFrame:
        frame = frame.sort_values("Date").copy()
        home_mask = frame["home_advantage"].astype(str).str.lower().eq("h")
        home_y = pd.to_numeric(col_or_nan(frame, "Home_Yellow"), errors="coerce")
        home_r = pd.to_numeric(col_or_nan(frame, "Home_Red"), errors="coerce")
        away_y = pd.to_numeric(col_or_nan(frame, "Away_Yellow"), errors="coerce")
        away_r = pd.to_numeric(col_or_nan(frame, "Away_Red"), errors="coerce")
        home_cards = home_y.fillna(0.0) + 2.0 * home_r.fillna(0.0)
        away_cards = away_y.fillna(0.0) + 2.0 * away_r.fillna(0.0)
        team_cards = np.where(home_mask, home_cards, away_cards)
        opp_cards = np.where(home_mask, away_cards, home_cards)
        pk_for = pd.to_numeric(col_or_nan(frame, "PK"), errors="coerce").fillna(0.0)
        pk_against = pd.to_numeric(
            col_or_nan(frame, "PK_Allowed"), errors="coerce"
        ).fillna(0.0)
        raw = pd.Series(
            (opp_cards - team_cards) + 2.0 * (pk_for - pk_against), index=frame.index
        ).fillna(0.0)
        std = raw.std()
        frame["_ref_signal_norm"] = (
            (raw - raw.mean()) / std if (pd.notna(std) and std > 0) else 0.0
        )
        return frame

    hist = history_df.copy()
    hist["_ref"] = (
        hist["Referee"].astype(str).str.lower().str.strip().replace("", "unknown")
    )
    hist = _compute_ref_signal(hist)
    hist["Referee_Bias_Score"] = (
        hist.groupby(["_ref", "Team"])["_ref_signal_norm"]
        .transform(lambda s: s.shift(1).expanding().mean())
        .fillna(0.0)
    )
    relegation_mask = hist["team_position"].fillna(10) >= 16
    title_mask = hist["team_position"].fillna(10) <= 2
    hist["Referee_Bias_Score"] *= np.where(relegation_mask | title_mask, 1.3, 1.0)

    bias_lookup = (
        hist[["_ref", "Team", "Date", "Referee_Bias_Score"]].sort_values("Date").copy()
    )
    out = df.copy()
    out["_ref"] = (
        out["Referee"].astype(str).str.lower().str.strip().replace("", "unknown")
    )
    out["Referee_Bias_Score"] = 0.0

    merged_parts = []
    for (ref, team), left_group in out.groupby(["_ref", "Team"], sort=False):
        right_group = bias_lookup[
            bias_lookup["_ref"].eq(ref) & bias_lookup["Team"].eq(team)
        ][["Date", "Referee_Bias_Score"]].sort_values("Date")
        left_group = left_group.sort_values("Date").copy()
        left_group["_orig_idx"] = left_group.index
        if right_group.empty:
            merged_parts.append(left_group)
            continue
        merged = pd.merge_asof(
            left_group,
            right_group.rename(columns={"Referee_Bias_Score": "_bias_new"}),
            on="Date",
            direction="backward",
            allow_exact_matches=False,
        )
        merged["Referee_Bias_Score"] = merged["_bias_new"].fillna(0.0)
        merged.drop(columns=["_bias_new"], inplace=True)
        merged_parts.append(merged)

    if merged_parts:
        out = (
            pd.concat(merged_parts)
            .sort_values("_orig_idx")
            .drop(columns=["_orig_idx", "_ref"], errors="ignore")
        )
    else:
        out.drop(columns=["_ref"], inplace=True, errors="ignore")

    log("Referee bias features attached")
    return out


def attach_match_context_features(
    df: pd.DataFrame, history_df: pd.DataFrame
) -> pd.DataFrame:
    out = df.sort_values("Date").copy()
    feature_rows = []
    for _, row in out.iterrows():
        past = history_df[history_df["Date"] < row["Date"]]
        recent_form = calculate_recent_form(past, row["Team"], row["Date"], window=5)
        opp_strength = calculate_opponent_strength(
            past, row["Opponent"], row["Date"], window=10
        )
        feats = _match_context_features_for_row(
            row, ROLLING_WINDOW, recent_form, opp_strength
        )
        feats["referee_pressure_impact"] = _referee_pressure_impact_for_row(row)
        feature_rows.append(feats)
    out = pd.concat([out, pd.DataFrame(feature_rows, index=out.index)], axis=1)
    log("Match-context features attached")
    return out


def attach_dynamic_motivation(
    df: pd.DataFrame, history_df: pd.DataFrame
) -> pd.DataFrame:
    out = df.sort_values("Date").copy()

    def _position_component(p):
        if pd.isna(p):
            return 0.0
        if p == 1:
            return 0.4
        if p <= 5:
            return 0.25
        if p <= 8:
            return 0.15
        if p >= 16:
            return 0.5
        return 0.0

    scores = []
    for _, row in out.iterrows():
        past = history_df[history_df["Date"] < row["Date"]]
        team = row["Team"]
        opp = row["Opponent"]
        team_pos = row.get("team_position", np.nan)
        opp_pos = row.get("opponent_position", np.nan)
        form = calculate_recent_form(past, team, row["Date"], window=5)
        form_score = 0.3 * form["points_per_game"] + 0.05 * max(form["streak"], -3)
        opp_strength = calculate_opponent_strength(past, opp, row["Date"], window=10)
        total = (
            form_score
            + _position_component(team_pos)
            + 0.15 * opp_strength
            + 0.2 * calculate_fixture_importance(team_pos, opp_pos)
            + (0.05 if str(row.get("home_advantage", "a")).lower() == "h" else 0.0)
            + TEAM_BOOST_OVERRIDES.get(team.lower(), 0.0)
        )
        scores.append(min(1.0, max(0.0, total)))

    out["Motivation_Score"] = scores
    log("Dynamic motivation scores calculated")
    return out

## 8. Split-Aware Feature Engineering Orchestrator

In [22]:
def engineer_split_features(
    split_df: pd.DataFrame,
    history_df: pd.DataFrame,
    comp_df: pd.DataFrame,
    pos_map: dict,
) -> pd.DataFrame:
    split_df = split_df.reset_index(drop=True).copy()
    history_df = history_df.reset_index(drop=True).copy()

    split_df["_is_split"] = True
    history_df["_is_split"] = False

    combined = (
        pd.concat([history_df, split_df])
        .drop_duplicates()
        .sort_values("Date")
        .reset_index(drop=True)
    )
    combined = build_rolling_features(combined)

    df = combined[combined["_is_split"] == True].copy().reset_index(drop=True)
    history_rolling = (
        combined[combined["_is_split"] == False].copy().reset_index(drop=True)
    )

    df = attach_fatigue(df, comp_df)
    df = attach_opponent_rolling_features(df, source_df=history_rolling)
    df = attach_position_features(df, pos_map)
    df = attach_h2h_features(df, history_df=history_rolling)
    df, _ = attach_context_features(df)

    history_rolling = attach_position_features(history_rolling, pos_map)
    df = attach_referee_features(df, history_df=history_rolling)
    df = attach_match_context_features(df, history_df=history_rolling)
    df = attach_dynamic_motivation(df, history_df=history_rolling)

    df = _drop_rolling_helper_columns(df)
    df.drop(columns=["_is_split"], inplace=True, errors="ignore")
    return df

## 9. Feature Column Selection and Leakage Assertion

In [23]:
def select_feature_columns(df: pd.DataFrame):
    window_cols = []
    for w in ROLLING_WINDOWS:
        window_cols.extend(
            [
                f"rolling_xG_{w}",
                f"rolling_xGA_{w}",
                f"rolling_xpts_{w}",
                f"rolling_ppda_{w}",
                f"rolling_scored_{w}",
                f"rolling_conceded_{w}",
                f"rolling_win_rate_{w}",
                f"rolling_goal_diff_{w}",
                f"weighted_xG_{w}",
                f"weighted_xGA_{w}",
                f"weighted_xpts_{w}",
                f"weighted_ppda_{w}",
                f"weighted_scored_{w}",
                f"weighted_conceded_{w}",
                f"weighted_win_rate_{w}",
                f"weighted_goal_diff_{w}",
                f"opp_rolling_xG_{w}",
                f"opp_rolling_xGA_{w}",
                f"opp_rolling_xpts_{w}",
                f"opp_rolling_ppda_{w}",
                f"opp_rolling_scored_{w}",
                f"opp_rolling_conceded_{w}",
                f"opp_rolling_win_rate_{w}",
                f"opp_rolling_goal_diff_{w}",
                f"opp_weighted_xG_{w}",
                f"opp_weighted_xGA_{w}",
                f"opp_weighted_xpts_{w}",
                f"opp_weighted_ppda_{w}",
                f"opp_weighted_scored_{w}",
                f"opp_weighted_conceded_{w}",
                f"opp_weighted_win_rate_{w}",
                f"opp_weighted_goal_diff_{w}",
            ]
        )

    base_numeric = [
        "fatigue_score",
        "fatigue_7d",
        "fatigue_14d",
        "fatigue_30d",
        "fatigue_recent_share",
        "team_position",
        "opponent_position",
        "position_gap",
    ]
    context_cols = [
        "Possession",
        "Shot_Creating_Actions",
        "SCA",
        "Successful_Dribbles",
        "Dribbles",
        "Final_Third_Entries",
        "Final_Third_Entries_Allowed",
        "Aerial_Battles_Won_Pct",
        "Aerial_Battles_Won%",
        "Save_Pct",
        "Save%",
        "PPDA",
        "Allowed_PPDA",
    ]
    psych_cols = ["Referee_Bias_Score", "Motivation_Score", "referee_pressure_impact"]
    interaction_cols = [
        "h2h_win_rate",
        "h2h_goal_diff",
        "h2h_match_count",
        "h2h_points_per_game",
        "form_vs_strength",
        "momentum",
        "xg_edge",
        "goal_diff_edge",
        "win_rate_edge",
    ]

    numeric_cols = [
        c
        for c in (
            base_numeric + window_cols + context_cols + psych_cols + interaction_cols
        )
        if c in df.columns
    ]
    cat_cols = ["home_advantage"]
    all_feat_cols = numeric_cols + cat_cols
    log(
        f"Features: {len(all_feat_cols)} total (numeric={len(numeric_cols)}, categorical={len(cat_cols)})"
    )
    return numeric_cols, cat_cols, all_feat_cols


def assert_no_perfect_target_leakage(X: pd.DataFrame, y: pd.Series) -> None:
    target = y.astype(str).str.lower().str.strip()
    target_codes = pd.Series(pd.Categorical(target).codes, index=y.index, dtype=float)
    leaked = []
    for col in X.columns:
        feat = X[col]
        if feat.dtype == object or pd.api.types.is_string_dtype(feat):
            if feat.astype(str).str.lower().str.strip().equals(target):
                leaked.append(col)
        elif pd.api.types.is_numeric_dtype(feat):
            num = pd.to_numeric(feat, errors="coerce")
            aligned = pd.DataFrame({"f": num, "t": target_codes}).dropna()
            if not aligned.empty and np.array_equal(
                np.sort(aligned["f"].unique()), np.sort(aligned["t"].unique())
            ):
                leaked.append(col)
    if leaked:
        raise ValueError("Leakage detected in: " + ", ".join(leaked))

## 10. Train/Validation/Test Matrix Assembly

In [24]:
base_df = load_premier_league(DATA_ROOT)
base_df = _normalize_base_columns(base_df)
base_df = _add_rolling_helper_columns(base_df)
base_df = _filter_trainable_rows(base_df)

raw_train, raw_val, raw_test = time_based_split(base_df, val_frac=0.20, test_frac=0.20)

comp_df = _load_competition_events(DATA_ROOT)
pos_map = load_position_map(DATA_ROOT / "league_position_after20.csv")

log("Engineering train features ...")
train_df = engineer_split_features(
    raw_train, history_df=raw_train, comp_df=comp_df, pos_map=pos_map
)

log("Engineering val features ...")
val_df = engineer_split_features(
    raw_val, history_df=raw_train, comp_df=comp_df, pos_map=pos_map
)

log("Engineering test features ...")
train_val = pd.concat([raw_train, raw_val]).sort_values("Date").reset_index(drop=True)
test_df = engineer_split_features(
    raw_test, history_df=train_val, comp_df=comp_df, pos_map=pos_map
)

numeric_cols, cat_cols, all_feat_cols = select_feature_columns(train_df)

for fdf in [val_df, test_df]:
    for c in all_feat_cols:
        if c not in fdf.columns:
            fdf[c] = np.nan

X_train, y_train = train_df[all_feat_cols], train_df["Result"]
X_val, y_val = val_df[all_feat_cols], val_df["Result"]
X_test, y_test = test_df[all_feat_cols], test_df["Result"]

assert_no_perfect_target_leakage(X_train, y_train)
log("Leakage check passed.")

[INFO] Loaded additional_data/premier_league.csv
[INFO] Raw rows loaded: 3,076
[INFO] Rows after filtering: 3,076
[INFO] Split: train=2,034  val=606  test=436
[INFO] Engineering train features ...
[INFO] Rolling features built (windows=(3, 5, 10))
[INFO] Fatigue scores attached from 920 competition rows
[INFO] Opponent rolling features attached
[INFO] H2H features attached
[INFO] Context features: ['Possession', 'Shot_Creating_Actions', 'Successful_Dribbles', 'Final_Third_Entries', 'Final_Third_Entries_Allowed', 'Aerial_Battles_Won_Pct', 'Save_Pct', 'PPDA', 'Allowed_PPDA']
[INFO] Referee bias features attached
[INFO] Match-context features attached
[INFO] Dynamic motivation scores calculated
[INFO] Engineering val features ...
[INFO] Rolling features built (windows=(3, 5, 10))
[INFO] Fatigue scores attached from 920 competition rows
[INFO] Opponent rolling features attached
[INFO] H2H features attached
[INFO] Context features: ['Possession', 'Shot_Creating_Actions', 'Successful_Dribble

## 11. Preprocessing Pipelines and Model Builders

In [25]:
def _base_preprocessor(num_feats, cat_feats):
    return ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), num_feats),
            (
                "cat",
                Pipeline(
                    [
                        ("imp", SimpleImputer(strategy="most_frequent")),
                        (
                            "onehot",
                            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                        ),
                    ]
                ),
                cat_feats,
            ),
        ]
    )


def _scaled_preprocessor(num_feats, cat_feats):
    return ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    [
                        ("imp", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                num_feats,
            ),
            (
                "cat",
                Pipeline(
                    [
                        ("imp", SimpleImputer(strategy="most_frequent")),
                        (
                            "onehot",
                            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                        ),
                    ]
                ),
                cat_feats,
            ),
        ]
    )


def build_logistic_regression_pipeline(n, c):
    return Pipeline(
        [
            ("preprocess", _scaled_preprocessor(n, c)),
            (
                "clf",
                LogisticRegression(
                    max_iter=3000,
                    class_weight="balanced",
                    random_state=42,
                    solver="lbfgs",
                ),
            ),
        ]
    )


def build_decision_tree_pipeline(n, c):
    return Pipeline(
        [
            ("preprocess", _base_preprocessor(n, c)),
            (
                "clf",
                DecisionTreeClassifier(
                    max_depth=12,
                    min_samples_leaf=5,
                    class_weight="balanced",
                    random_state=42,
                ),
            ),
        ]
    )


def build_random_forest_pipeline(n, c, n_estimators=200):
    return Pipeline(
        [
            ("preprocess", _base_preprocessor(n, c)),
            (
                "clf",
                RandomForestClassifier(
                    n_estimators=n_estimators,
                    class_weight="balanced",
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ]
    )


def build_svm_pipeline(n, c):
    return Pipeline(
        [
            ("preprocess", _scaled_preprocessor(n, c)),
            (
                "clf",
                SVC(
                    kernel="rbf",
                    C=1.0,
                    gamma="scale",
                    class_weight="balanced",
                    random_state=42,
                ),
            ),
        ]
    )


def build_mlp_pipeline(n, c):
    return Pipeline(
        [
            ("preprocess", _scaled_preprocessor(n, c)),
            (
                "clf",
                MLPClassifier(
                    hidden_layer_sizes=(64, 32),
                    activation="relu",
                    alpha=1e-4,
                    learning_rate_init=1e-3,
                    max_iter=600,
                    random_state=42,
                ),
            ),
        ]
    )


def build_xgboost_pipeline(n, c, n_estimators=200):
    return Pipeline(
        [
            ("preprocess", _base_preprocessor(n, c)),
            (
                "clf",
                XGBClassifier(
                    n_estimators=n_estimators,
                    max_depth=6,
                    learning_rate=0.05,
                    subsample=0.9,
                    colsample_bytree=0.9,
                    objective="multi:softprob",
                    eval_metric="mlogloss",
                    random_state=42,
                ),
            ),
        ]
    )

## 12. Model Training, Evaluation Reports, and Ranked Results Table

In [26]:
# Potential CV-loop issues to be aware of:
print("Potential CV issues to watch:")
print("1) Feature engineering must be rebuilt per fold from raw split data.")
print("2) Validation fold must only see history before fold start date.")
print("3) XGBoost label encoder must be fit inside each fold only.")
print("4) Do not shuffle temporal splits.")

WDL_LABELS = ["w", "d", "l"]
N_CV_SPLITS = 5


def make_walk_forward_date_folds(df: pd.DataFrame, n_splits: int = N_CV_SPLITS):
    ordered = df.copy()
    ordered["Date"] = pd.to_datetime(ordered["Date"], errors="coerce")
    ordered = (
        ordered[ordered["Date"].notna()].sort_values("Date").reset_index(drop=True)
    )

    unique_dates = np.array(sorted(ordered["Date"].unique()))
    if len(unique_dates) <= n_splits:
        raise ValueError(
            f"Need more unique dates ({len(unique_dates)}) than n_splits ({n_splits})"
        )

    tscv = TimeSeriesSplit(n_splits=n_splits)
    folds = []
    for fold_idx, (train_date_idx, val_date_idx) in enumerate(
        tscv.split(unique_dates), start=1
    ):
        train_dates = set(unique_dates[train_date_idx])
        val_dates = set(unique_dates[val_date_idx])

        train_fold_raw = (
            ordered[ordered["Date"].isin(train_dates)]
            .copy()
            .sort_values("Date")
            .reset_index(drop=True)
        )
        val_fold_raw = (
            ordered[ordered["Date"].isin(val_dates)]
            .copy()
            .sort_values("Date")
            .reset_index(drop=True)
        )

        if train_fold_raw.empty or val_fold_raw.empty:
            continue

        train_end = pd.to_datetime(train_fold_raw["Date"]).max()
        val_start = pd.to_datetime(val_fold_raw["Date"]).min()
        if train_end >= val_start:
            raise ValueError(
                f"Fold {fold_idx} overlap detected: train_end={train_end}, val_start={val_start}"
            )

        folds.append((fold_idx, train_fold_raw, val_fold_raw))
    return folds


def evaluate_model_on_fold(
    model_name: str,
    model_builder,
    train_fold_raw: pd.DataFrame,
    val_fold_raw: pd.DataFrame,
    comp_df: pd.DataFrame,
    pos_map: dict,
    n_estimators: int = 200,
):
    train_feat = engineer_split_features(
        train_fold_raw, history_df=train_fold_raw, comp_df=comp_df, pos_map=pos_map
    )
    val_feat = engineer_split_features(
        val_fold_raw, history_df=train_fold_raw, comp_df=comp_df, pos_map=pos_map
    )

    numeric_cols_fold, cat_cols_fold, all_feat_cols_fold = select_feature_columns(
        train_feat
    )
    for c in all_feat_cols_fold:
        if c not in val_feat.columns:
            val_feat[c] = np.nan

    X_train_fold = train_feat[all_feat_cols_fold]
    y_train_fold = train_feat["Result"]
    X_val_fold = val_feat[all_feat_cols_fold]
    y_val_fold = val_feat["Result"]

    assert_no_perfect_target_leakage(X_train_fold, y_train_fold)

    if model_name == "XGBoostClassifier":
        le_fold = LabelEncoder()
        y_train_enc = le_fold.fit_transform(y_train_fold)
        y_val_enc = le_fold.transform(y_val_fold)
        model = model_builder(
            numeric_cols_fold, cat_cols_fold, n_estimators=n_estimators
        )
        model.fit(X_train_fold, y_train_enc)
        y_pred_enc = model.predict(X_val_fold)
        y_pred = le_fold.inverse_transform(np.asarray(y_pred_enc, dtype=int))
        accuracy = accuracy_score(y_val_fold, y_pred)
        f1_w, f1_d, f1_l = f1_score(
            y_val_fold, y_pred, labels=WDL_LABELS, average=None, zero_division=0
        )

        fold_log_loss = np.nan
        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(X_val_fold)
            fold_log_loss = log_loss(
                y_val_enc, proba, labels=np.arange(len(le_fold.classes_))
            )
    else:
        model = model_builder(numeric_cols_fold, cat_cols_fold)
        model.fit(X_train_fold, y_train_fold)
        y_pred = model.predict(X_val_fold)
        accuracy = accuracy_score(y_val_fold, y_pred)
        f1_w, f1_d, f1_l = f1_score(
            y_val_fold, y_pred, labels=WDL_LABELS, average=None, zero_division=0
        )

        fold_log_loss = np.nan
        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(X_val_fold)
            fold_log_loss = log_loss(y_val_fold, proba, labels=list(model.classes_))

    return {
        "accuracy": accuracy,
        "f1_w": f1_w,
        "f1_d": f1_d,
        "f1_l": f1_l,
        "log_loss": fold_log_loss,
        "train_rows": len(train_fold_raw),
        "val_rows": len(val_fold_raw),
    }


train_val_raw = (
    pd.concat([raw_train, raw_val], axis=0).sort_values("Date").reset_index(drop=True)
)
cv_folds = make_walk_forward_date_folds(train_val_raw, n_splits=N_CV_SPLITS)

cv_model_specs = [
    ("LogisticRegression", build_logistic_regression_pipeline),
    ("DecisionTreeClassifier", build_decision_tree_pipeline),
    ("RandomForestClassifier", build_random_forest_pipeline),
    ("SVM", build_svm_pipeline),
    ("MLPClassifier", build_mlp_pipeline),
    ("XGBoostClassifier", build_xgboost_pipeline),
]

cv_records = []
for fold_idx, train_fold_raw, val_fold_raw in cv_folds:
    print(f"\n=== Fold {fold_idx}/{len(cv_folds)} ===")
    for model_name, model_builder in cv_model_specs:
        metrics = evaluate_model_on_fold(
            model_name=model_name,
            model_builder=model_builder,
            train_fold_raw=train_fold_raw,
            val_fold_raw=val_fold_raw,
            comp_df=comp_df,
            pos_map=pos_map,
            n_estimators=200,
        )
        cv_records.append(
            {
                "fold": fold_idx,
                "model": model_name,
                **metrics,
            }
        )
        print(
            f"{model_name:22s} acc={metrics['accuracy']:.4f} "
            f"f1(w/d/l)=({metrics['f1_w']:.3f}/{metrics['f1_d']:.3f}/{metrics['f1_l']:.3f}) "
            f"logloss={metrics['log_loss'] if pd.notna(metrics['log_loss']) else 'NA'}"
        )

cv_results_df = pd.DataFrame(cv_records)

cv_summary_df = (
    cv_results_df.groupby("model", as_index=False)
    .agg(
        mean_accuracy=("accuracy", "mean"),
        std_accuracy=("accuracy", "std"),
        mean_f1_w=("f1_w", "mean"),
        mean_f1_d=("f1_d", "mean"),
        mean_f1_l=("f1_l", "mean"),
        mean_log_loss=("log_loss", "mean"),
    )
    .sort_values("mean_accuracy", ascending=False)
    .reset_index(drop=True)
)

print("\nCV Accuracy Summary (mean +/- std):")
print(cv_summary_df[["model", "mean_accuracy", "std_accuracy"]].to_string(index=False))

print("\nCV Mean Per-Class F1 (W, D, L):")
print(
    cv_summary_df[["model", "mean_f1_w", "mean_f1_d", "mean_f1_l"]].to_string(
        index=False
    )
)

# Final held-out test evaluation after CV
print("\nRetraining on full train+val and evaluating on held-out test set...")
final_train_feat = engineer_split_features(
    train_val_raw, history_df=train_val_raw, comp_df=comp_df, pos_map=pos_map
)
final_test_feat = engineer_split_features(
    raw_test, history_df=train_val_raw, comp_df=comp_df, pos_map=pos_map
)

final_numeric_cols, final_cat_cols, final_all_feat_cols = select_feature_columns(
    final_train_feat
)
for c in final_all_feat_cols:
    if c not in final_test_feat.columns:
        final_test_feat[c] = np.nan

X_train_final = final_train_feat[final_all_feat_cols]
y_train_final = final_train_feat["Result"]
X_test_final = final_test_feat[final_all_feat_cols]
y_test_final = final_test_feat["Result"]

assert_no_perfect_target_leakage(X_train_final, y_train_final)

final_results = []
for model_name, model_builder in cv_model_specs:
    if model_name == "XGBoostClassifier":
        le_final = LabelEncoder()
        y_train_final_enc = le_final.fit_transform(y_train_final)
        model = model_builder(final_numeric_cols, final_cat_cols, n_estimators=200)
        model.fit(X_train_final, y_train_final_enc)
        y_test_pred = le_final.inverse_transform(
            np.asarray(model.predict(X_test_final), dtype=int)
        )
    else:
        model = model_builder(final_numeric_cols, final_cat_cols)
        model.fit(X_train_final, y_train_final)
        y_test_pred = model.predict(X_test_final)

    final_acc = accuracy_score(y_test_final, y_test_pred)
    final_results.append({"model": model_name, "final_test_accuracy": final_acc})
    print(f"{model_name:22s} final_test_accuracy={final_acc:.4f}")

final_results_df = (
    pd.DataFrame(final_results)
    .sort_values("final_test_accuracy", ascending=False)
    .reset_index(drop=True)
)

print("\nFinal Held-Out Test Accuracy (trained on full train+val):")
print(final_results_df.to_string(index=False))

cv_results_df, cv_summary_df, final_results_df

Potential CV issues to watch:
1) Feature engineering must be rebuilt per fold from raw split data.
2) Validation fold must only see history before fold start date.
3) XGBoost label encoder must be fit inside each fold only.
4) Do not shuffle temporal splits.

=== Fold 1/5 ===
[INFO] Rolling features built (windows=(3, 5, 10))
[INFO] Fatigue scores attached from 920 competition rows
[INFO] Opponent rolling features attached
[INFO] H2H features attached
[INFO] Context features: ['Possession', 'Shot_Creating_Actions', 'Successful_Dribbles', 'Final_Third_Entries', 'Final_Third_Entries_Allowed', 'Aerial_Battles_Won_Pct', 'Save_Pct', 'PPDA', 'Allowed_PPDA']
[INFO] Referee bias features attached
[INFO] Match-context features attached
[INFO] Dynamic motivation scores calculated
[INFO] Rolling features built (windows=(3, 5, 10))
[INFO] Fatigue scores attached from 920 competition rows
[INFO] Opponent rolling features attached
[INFO] H2H features attached
[INFO] Context features: ['Possession', 

(    fold                   model  accuracy      f1_w      f1_d      f1_l  \
 0      1      LogisticRegression  0.403756  0.473333  0.307692  0.422939   
 1      1  DecisionTreeClassifier  0.382629  0.469136  0.127660  0.441176   
 2      1  RandomForestClassifier  0.434272  0.506579  0.158416  0.531792   
 3      1                     SVM  0.401408  0.473868  0.228137  0.483444   
 4      1           MLPClassifier  0.410798  0.506173  0.151515  0.472727   
 5      1       XGBoostClassifier  0.410798  0.450704  0.211538  0.494444   
 6      2      LogisticRegression  0.418103  0.458824  0.298851  0.470948   
 7      2  DecisionTreeClassifier  0.377155  0.385852  0.249027  0.461111   
 8      2  RandomForestClassifier  0.478448  0.543081  0.123457  0.563969   
 9      2                     SVM  0.463362  0.508475  0.301255  0.531343   
 10     2           MLPClassifier  0.424569  0.482955  0.241379  0.488372   
 11     2       XGBoostClassifier  0.433190  0.491525  0.238532  0.494382   